<font color=#009afa size=50>DAT-6002 - Artificial Intelligence</font><br>
<font size=50>WRIT1 Part 1 - Semantic Networks</font><br>
<font size=20 color=#00c382>Medieval Weapons</font><br>

|**Assessment Code:** |WRIT1|
|---|---|
|**Author:**|Tom Rowan|
|**Course:**|DAT6002_S2_25|
|**Course Tutor:**|Imtiaz Hussain Khan, PhD|


---

# Initialise Notebook

In [ ]:
# Install Requirements Using Pip
%pip install -r requirements.txt

### Import Python Libraries

In [3]:
# Standard Libraries
import sys
import json
from datetime import datetime
import re

# Requests
import requests

# Pandas
import pandas as pd

# Colorama
from colorama import Fore, Back, Style

# NetworkX
import networkx as nx

# PyVis
from pyvis.network import Network

# Do not print warnings!
import warnings
warnings.filterwarnings('ignore')

### Create a Timestamp

In [4]:
# Generate Timestamp
today = pd.Timestamp(datetime.today())
now_date_timestamp = str(today.strftime("%Y%m%d"))
now_datetime_timestamp = str(today.strftime("%Y%m%d-%H%M"))

### Functions for Pretty Output

print_output - Print coloured Output

In [5]:
# Function to print coloured output
def print_output (str1, str2, delim=':', colour='yellow'):
    if colour.upper() in dir(Fore):
        colour = colour.upper()
    else:
        colour = 'YELLOW'
    print (f'{getattr(Fore, colour)}{str(str1)}{delim}{Style.RESET_ALL} {str(str2)}')

### Check for Internet Access

In [6]:
_urls_to_check = ['http://connectivitycheck.gstatic.com/generate_204',
                  'https://www.github.com',
                  'https://datascience.daemonchild.com']

Set this to False if you have a local dataset

In [7]:
internet_required = False

In [8]:
# Function to Safely Check for Internet Connectivity
def check_internet (url_list):
    online = False
    for _url in url_list:
        try:
            response = requests.get(_url, timeout=3)
            if response.status_code == 200:
                online = True
                break
        except requests.RequestException:
            output_string = f"Check Internet failed for {_url}"
            print_output ('[FAIL]',output_string, colour='red', delim='')
    return online

# Perform the checks
if check_internet (_urls_to_check ):
    print_output ('[INTERNET]','Internet Connectivity Detected. Using Internet resources.',colour='green', delim='')
    online = True

else:
    print ()
    print_output ('[INTERNET]','No Internet Connectivity.', colour='red', delim='')
    online = False


# Stop running if Internet is required
if online == False & internet_required == True:
    sys.exit('Stopping notebook run here.')    


[INTERNET] Internet Connectivity Detected. Using Internet resources.


---
# Semantic Network Class

This class defines a semantic network.

It allows the user to add nodes and edges to the graph as ‘is_a’ and ‘has_a’ relationships.<br> 
From the resulting network, the user can output the network into JSON and Prolog files.<br>
They can also create graphical visualisations automatically.<br>

It also allows the creation of a network from an existing Prolog file, as well as suitably formatted CSV and JSON files.

Internally it uses NetworkX to model the graph.

## Methods 
### Initialisation

The user must provide a root_node and a title for the network.

```weapons_net = SemanticNetwork(root_node="weapon", title="Medieval Weapons")```


### add_is_a

Adds an is_a relationship to the network.<br>
Parameters: node, parent


### add_has_a

Adds a has_a property to the network.<br>
Parameters: node, property, value

### export_to_json

Exports the network to JSON.<br>
Parameters: filename


### import_from_json

Imports the network from JSON. This destroys any existing content.<br>
Parameters: filename


### export_to_prolog

Exports the network to Prolog.<br>
A rules file must be included to add rules to the network.<br>
Parameters: filename, rules_filename


### import_from_prolog

Imports the network from a Prolog file. <br>
This destroys any existing content, and ignores any rules.<br>
Parameters: filename


### draw_hierarchical
Draws a hierarchical network diagram, exported into an HTML file.<br>
The root is at the top, and nodes are positioned in a horizonal tree format.<br>
This uses PyVis, which requires Internet access in order to download JavaScript to make the HTML file work.<br>
Can optionally show the levels of each node on the diagram.<br>
You can also pass an alternative palette.<br>
Parameters: filename, show_levels, [palette]

### draw_dynamic
Draws a network diagram, exported into an HTML file.<br>
Nodes will attempt to optimally reposition themselves if dragged or moved.<br>
This uses PyVis, which requires Internet access in order to download JavaScript to make the HTML file work.<br>
Can optionally show the levels of each node on the diagram.<br>
You can also pass an alternative palette.<br>
Parameters: filename, show_levels, [palette]



## Building a network

The network can be populated manually, for example:

```
# Add weapons to the hierarchy

# Bladed Hand Weapon Categories
weapons_net.add_is_a("sword", "bladed_hand_weapon")

# Properties of a Generic Sword
# Read as follows: a sword has_a (has_part) pommel
weapons_net.add_has_a("sword", "has_part", "pommel")
weapons_net.add_has_a("sword", "has_part", "hilt")
weapons_net.add_has_a("sword", "has_part", "blade")
weapons_net.add_has_a("sword", "causes_damage", "cut")
weapons_net.add_has_a("sword", "primary_material", "steel" )
```


In [9]:
# Define Class SemanticNetwork
class SemanticNetwork:

    # Internal functions are prefixed with an underscore to indicate they are private.

    # Constructor
    def __init__(self, root_node="RootNode",title = "A Semantic Network"):

        # Create graph object
        self.G = nx.DiGraph()
        self.root = root_node
        
        # Descriptive title for the network
        self.title = title

        # Default Palette
        # Based on https://coolors.co
        palette =   {'bg':"#000000", 
                     'text': "#ffffff", 
                     'edge_label': "#F0D90A", 
                     'edge_struct': "#c4830b81", 
                     'edge_prop': "#7847ff81",
                     0:"#C3ADFF", 
                     1:"#9670ff", 
                     2:"#7847ff", 
                     3:"#5A1FFF", 
                     4:"#4100F5", 
                     5:"#3100B8", 
                     6:"#22007C"}

        self.palette = palette
    # end constructor

    #
    # ***** Add to Network *****
    #

    # Add relationships to the network (IS-A type)
    def add_is_a(self, node, parent):
        self.G.add_edge(node, parent, label="is_a")


    # Add properties to the network (HAS-A type)
    def add_has_a(self, node, property, value):
        self.G.add_edge(node, value, label=property)


    # Calculates distance from the root node
    # Used for dynamic scaling and choosing colours from palette
    def _get_node_level(self, node):
        try:
            # Empty list to store filtered edges
            hierarchy_only_edges = []

            # Iterate through every edge in the graph
            for source, target, data in self.G.edges(data=True):

                # We only want 'is_a' (structural) links, not 'has_a' (properties)
                # This prevents property nodes from making short cut paths!
                edge_type = data.get('label')
                
                if edge_type == 'is_a':
                    # Add this connection to list
                    hierarchy_only_edges.append((source, target))

            # Create a temporary directed graph using the filtered edges
            structural_subgraph = nx.DiGraph(hierarchy_only_edges)

            # Calculate the shortest path distance to the root
            distance = nx.shortest_path_length(
                structural_subgraph, 
                source=node,
                target=self.root
            )
            
            return distance

        except Exception:
            # Returns None if the node isn't part of the hierarchy or if there is no path to the root.
            return None
        

    #
    # ***** JSON *****
    #

    # Export Network to json file 
    def export_to_json(self, filename):

        data = nx.node_link_data(self.G)    # NB deprecated, but suggested 'edges' replacement fails...
        # data = nx.edges(self.G)           # This is the suggested replacement.

        try: 
            with open(filename, 'w') as f:
                json.dump(data, f, indent=4)

            print_output(f"Successfully saved", f"{len(self.G.nodes)} nodes to {filename}")
        except Exception:
            print_output(f"Failed",f"saving to {filename}", colour='red')


    # Import Network from json file 
    def import_from_json(self, filename):
        
        try:
            with open(filename, 'r') as f:
                data = json.load(f)

            # NB this is destructive
            self.G = nx.node_link_graph(data)
            print_output(f"Successfully loaded", f"{len(self.G.nodes)} nodes from {filename}")

        except Exception:
            print_output(f"Failed",f"saving to {filename}", colour='red')

    #
    # ***** Prolog *****
    #
    
    def _get_parent_from_fact(self, fact_string):

        # This splits the string at the comma, takes the second half, 
        # then strips spaces and the ').'
        # Example: "is_a( springald, stone_thrower )." -> "stone_thrower"

        try:
            return fact_string.split(',')[1].split(')')[0].strip()
        
        except IndexError:
            # Fallback if string isn't as expected
            return fact_string 


    # The network can export itself to Prolog
    # Requires a rules file to be appended to the output
    def export_to_prolog (self, filename="prolog_file.pl",rules_filename='rules.pl'):

        # Separate storage for different types of facts
        is_a_facts = []
        has_a_facts = []
        
        # Empty file text
        prolog_file = []

        # Loop through the network
        for source, target, data in self.G.edges(data=True):
            label = data.get('label')

            # Fix any items that have been defined with space
            # Also ensure everything is lower case
            src = source.lower().replace(' ','_')
            tgt = target.lower().replace(' ','_')
            label = label.lower().replace(' ','_')
                
            # Grouping types together
            if label == 'is_a':

                # Append the level to the is_a fact in a tuple for sorting output file
                level = self._get_node_level(target)
                if level == None:
                    level = 99

                if src == self.root: 
                    level = 0

                # Add to the list
                is_a_facts.append((level, f'is_a( {src}, {tgt} ).'))

            else:
                # It's a has_a fact.

                # There is a special case when there is a 'from x to y' value (eg: 2,3 --> (2,3))
                if ',' in tgt:
                    tgt = f"({tgt})"

                # Add to the list
                has_a_facts.append(f"has_a( {src}, {label},{tgt} ).")   

        # Sorts by the first element of the tuple (level)
        is_a_facts.sort(key=lambda x: (x[0], self._get_parent_from_fact(x[1])))

        # Clean the tuples back into strings
        is_a_output = [item[1] for item in is_a_facts] 

        # Straightforward Alphabetical sort for has_a facts
        has_a_facts.sort()

        # Construct the file content in blocks
        
        prolog_file.append("%")
        prolog_file.append(f"% ***** {self.title} in Prolog *****")
        prolog_file.append("%")
        prolog_file.append(f"% Root_node is '{self.root}'")
        prolog_file.append("%")
        
        prolog_file.append("\n%")
        prolog_file.append("% ***** IS-A Facts")
        prolog_file.append("%\n")
        prolog_file.extend(is_a_output)

        prolog_file.append("\n%")
        prolog_file.append("% ***** HAS-A Facts")
        prolog_file.append("%\n")
        prolog_file.extend(has_a_facts)

        # Append the rules
        with open(rules_filename, 'r') as f:
            rules_data = f.read()

        prolog_file.append(rules_data)

        # Save to file
        with open(filename, 'w') as f:
            for line in prolog_file:
                f.write(str(line)+'\n')
        
        print_output("Wrote File",f"Prolog written to {filename}.")
        print(f" - wrote {str(len(is_a_output))} IS_A facts")
        print(f" - wrote {str(len(has_a_facts))} HAS_A facts")


    
    # The network can be imported from a prolog file provided it follows the right structure
    #
    #   File must be formatted as follows:
    #       is_a(mammal, animal)
    #       is_a(canine, mammal)
    #       ..
    #       has_a(canine, makes_noise, bark)
    #       has_a(canine, makes_noise, growl)
    #       has_a(canine, legs,4)
    #
    def import_from_prolog (self, filename='prolog_file.pl'):

        # Empty lists
        is_a_list = []
        has_a_list = []

        # Regex patterns
        is_a_pattern = r"^is_a\(\s*(\w+),\s*(\w+)\s*\)\."
        has_a_pattern = r"^has_a\(\s*(\w+),\s*(\w+),\s*(.*?)\s*\)\."

        with open(filename, 'r') as file:
            data = file.read()

            # Extracting is_a facts
            is_a_list = re.findall(is_a_pattern, data, re.MULTILINE)
            
            # Extracting has_a facts
            has_a_list = re.findall(has_a_pattern, data, re.MULTILINE)

        # Add elements to the network
        for fact in is_a_list:
            entity = fact[0]
            parent = fact[1]
            self.add_is_a(entity, parent)
    
        for fact in has_a_list:
            entity = fact[0]
            property = fact[1]
            value = fact[2]
            self.add_has_a(entity, property, value)

        print_output ("Imported network from prolog file", filename)
        print(f" - Added {str(len(is_a_list))} IS_A facts")
        print(f" - Added {str(len(has_a_list))} HAS_A facts")

    #
    # ***** Visualisation *****
    #

    # Create hierarchical visualisation
    def draw_hierarchical(self, filename, show_levels=False, filter_list=None, **kwargs):

        # Handle Additional Arguments
        palette = kwargs.get('palette', self.palette)

        # Initiate the PyVis Network
        net = Network(height="1000px", width="100%", bgcolor=palette['bg'], font_color=palette['text'], directed=True)

        # If a filter_list is supplied then only use those nodes in the diagram
        if filter_list:
            filtered_nodes = set()
            for node in filter_list:
                if node in self.G:
                    filtered_nodes.add(node)

                    # Get all ancestors and descendants (parents, grandparents, etc.)
                    # It's not the wrong way round, the tree root is at the top!
                    ancestors = nx.descendants(self.G, node)
                    filtered_nodes.update(ancestors)

                # Use only 'is_a' edges that connect our filtered nodes
                entity_nodes = {n for n in filtered_nodes if any(
                    data.get('label') == 'is_a' for source, target, data in self.G.edges(n, data=True)
                ) or n == self.root}
        else:
            # Draw all nodes that are part of the structural 'is_a' tree
            entity_nodes = set()
            for source, target, data in self.G.edges(data=True):
                if data.get('label') == 'is_a':
                    entity_nodes.add(source)
                    entity_nodes.add(target)
        
        # Sort the nodes by their depth level, then alphabetically. 
        sorted_nodes = sorted(list(entity_nodes), key=lambda x: (self._get_node_level(x) or 99, x))

        # Add Nodes with Property Lists (like a UML diagram)
        for node in sorted_nodes:
            props = []
            for _, target, data in self.G.out_edges(node, data=True):
                label = data.get('label')
                if label != 'is_a':
                    props.append(f"{label}: {target}")
            
            # Build the text for the node box label. 
            # If there are properties, list them under the node name. 
            # If not, just show the node name.
            if props:
                props.sort()
                prop_text = "\n".join(props)
                display_label = f"{node.upper()}\n{'—'*len(node)}\n{prop_text}"
            else:
                display_label = f"{node.upper()}"

            # Calculate the nodes level
            level = self._get_node_level(node)
            if level is None: level = 99
            if node == self.root: level = 0
            
            # Dynamic colouring based on level
            color = palette.get(level, "#D3D3D3")

            # Are we showing the levels in the label?
            if show_levels:
                node_label = f'[{level}] {display_label}'
            else:
                node_label = f'{display_label}'

            # Format and add the node to the network
            net.add_node(
                node, 
                label=node_label,
                shape="box",
                color=color,
                level=level, 
                font={'face': 'Arial', 'align': 'left', 'color': palette['text'], 'size': 40}
            )

        # Add edges, but only the 'is_a' edges that connect the (possibly filtered) nodes
        for source, target, data in self.G.edges(data=True):
    
            edge_type = data.get('label')            
            if edge_type == 'is_a':
                
                # Are the nodes part of the set?
                if source in entity_nodes and target in entity_nodes:
                    
                    net.add_edge(
                            source, 
                            target, 
                            width=3, 
                            color=palette['edge_struct']
                    )

        # Define graph options
        options = {
            "layout": {
                "hierarchical": {
                    "enabled": True,
                    "direction": "UD",            # Up-Down (Level 0 at the top)
                    "sortMethod": "directed",
                    "shakeTowards": "roots",      # Pulls loose branches tightly to their parents
                    "levelSeparation": 200,       
                    "nodeSpacing": 300,           
                    "treeSpacing": 300,           
                    "parentCentralization": True, 
                    "blockShifting": True,
                    "edgeMinimization": True      
                }
            },
            "physics": {
                "enabled": False                   # Or it will try to rearrange itself and mess up the hierarchy!
            },
            "edges": {
                "smooth": {
                    "type": "cubicBezier",        
                    "forceDirection": "vertical", 
                    "roundness": 0.4
                },
                "arrows": {
                    "to": {"enabled": True}
                }
            }
        }
        
         # Convert options for the visualisation JavaScript
        net.set_options(json.dumps(options))

        # Write out the diagram to the specified file
        net.write_html(filename)
        print_output("Wrote File",f"Network diagram written to {filename}.")     


    # Create a dynamic visualisation of the network using PyVis, saved as HTML file
    def draw_dynamic(self, filename, show_levels=False, **kwargs):

        # Handle Additional Arguments
        palette = kwargs.get('palette', self.palette)

        # Initiate the PyVis Network
        net = Network(height="1000", width="100%", bgcolor=palette['bg'], font_color=palette['text'], directed=True)
        
        # Add Nodes 
        for node in self.G.nodes():

            level = self._get_node_level(node) or 99
            if node == self.root:
                level = 0
            
            color = palette.get(level, "#D3D3D3")

            # Dynamic Scaling of nodes and fonts
            if level==0:
                dynamic_node_size = 120
                dynamic_font_size = 120
            else:
                dynamic_font_size = max(80 - (level * 8), 25) if level != 99 else 20
                dynamic_node_size = max(70 - (level * 7), 15) if level != 99 else 10

            # Add the level to the label
            if show_levels and level != 99:
                label = f'{node} (level {level})'
            else:
                label = node
            
            net.add_node(
                node, 
                label=label,
                color=color, 
                size = dynamic_node_size,
                font={'size': dynamic_font_size, 'face': 'arial', 'color': palette['text']}
            )

        # Add Edges
        for source, target, data in self.G.edges(data=True):
            label = data.get('label')
        
            net.add_edge(
                    source, 
                    target, 
                    label=label,
                    width=5 if label == 'is_a' else 2,
                    color= palette['edge_struct'] if label == 'is_a' else palette['edge_prop'],
                    font={'color': palette['edge_label'], 'size': 20, 'strokeWidth': 0}
            )

        # Define options for network
        options = {
            "nodes": {
                "font": {
                    "size": 30 
                }
            },
            "edges": {
                "font": {
                    "size": 18,
                    "strokeWidth": 0
                },
                "color": {"inherit": False}
            },
            "physics": {
                "enabled": True,
                "barnesHut": {
                    "gravitationalConstant": -15000, # Push nodes apart
                    "springLength": 400 # Lengthen the edges
                },
                "solver": "barnesHut"
            },
            "interaction": {
                "hideEdgesOnDrag": False,
                "hideEdgesOnZoom": False,
                "zoomView": True
            }
        }
        
        # Convert options for the visualisation JavaScript
        net.set_options(json.dumps(options))

        # Write out the diagram to the specified file
        net.write_html(filename)
        print_output("Wrote File",f"Network diagram written to {filename}.")


## Semantic Network - Medieval Weapons 

In [10]:
# Create new network object (Level 0)

weapons_net = SemanticNetwork(root_node='weapon', title="Medieval Weapons Semantic Network")
weapons_net.add_has_a('weapon', 'era_used', '476,1500')

# Add weapons to the hierarchy manually

### Level 1 - Categories
weapons_net.add_is_a('melee_weapon', 'weapon')
weapons_net.add_has_a('melee_weapon', 'effective_against', 'personnel')
weapons_net.add_has_a('melee_weapon', 'attack_rate', 'fast')
weapons_net.add_has_a('melee_weapon', 'movement_speed', 'fast')
weapons_net.add_has_a('melee_weapon', 'use_case', 'attack')
weapons_net.add_has_a('melee_weapon', 'ammunition_type', 'no_ammunition')
weapons_net.add_has_a('melee_weapon', 'is_throwable', 'not_throwable')
weapons_net.add_has_a('melee_weapon', 'people_reqd', 'one_person')

weapons_net.add_is_a('ranged_weapon', 'weapon')
weapons_net.add_has_a('ranged_weapon', 'effective_against', 'personnel')
weapons_net.add_has_a('ranged_weapon', 'attack_rate', 'medium')
weapons_net.add_has_a('ranged_weapon', 'movement_speed', 'fast')
weapons_net.add_has_a('ranged_weapon', 'use_case', 'attack')
weapons_net.add_has_a('ranged_weapon', 'is_throwable', 'not_throwable')
weapons_net.add_has_a('ranged_weapon', 'people_reqd', 'one_person')

weapons_net.add_is_a('siege_weapon', 'weapon')
weapons_net.add_has_a('siege_weapon', 'movement_speed', 'slow')
weapons_net.add_has_a('siege_weapon', 'use_case', 'attack')
weapons_net.add_has_a('siege_weapon', 'is_throwable', 'not_throwable')
weapons_net.add_has_a('siege_weapon', 'people_reqd', 'a_team')
weapons_net.add_has_a('siege_weapon', 'has_edges', 'no_edges')

### Level 2 - Categories
# Melee Weapon Categories
weapons_net.add_is_a('bladed_hand_weapon', 'melee_weapon')
weapons_net.add_has_a('bladed_hand_weapon', 'effective_range', 'close_quarters')
weapons_net.add_has_a('bladed_hand_weapon', 'attack_type', 'cut')
weapons_net.add_has_a('bladed_hand_weapon', 'has_part', 'blade')

weapons_net.add_is_a('blunt_hand_weapon', 'melee_weapon')
weapons_net.add_has_a('blunt_hand_weapon', 'effective_range', 'close_quarters')
weapons_net.add_has_a('blunt_hand_weapon', 'attack_type', 'bash')
weapons_net.add_has_a('blunt_hand_weapon', 'primary_material', 'wood')

weapons_net.add_is_a('polearm', 'melee_weapon')
weapons_net.add_has_a('polearm', 'effective_range', 'short')
weapons_net.add_has_a('polearm', 'attack_type', 'thrust')

# Ranged Weapon Categories
weapons_net.add_is_a('artillery', 'ranged_weapon')
weapons_net.add_has_a('artillery', 'effective_against', 'fortifications')
weapons_net.add_has_a('artillery', 'effective_range', 'long')
weapons_net.add_has_a('artillery', 'attack_rate', 'slow')
weapons_net.add_has_a('artillery', 'movement_speed', 'slow')
weapons_net.add_has_a('artillery', 'people_reqd', 'a_team')

weapons_net.add_is_a('bow', 'ranged_weapon')
weapons_net.add_has_a('bow', 'effective_range', 'medium')

weapons_net.add_is_a('francisca', 'ranged_weapon')
weapons_net.add_has_a('francisca', 'effective_range', 'short')
weapons_net.add_has_a('francisca', 'attack_rate', 'fast')
weapons_net.add_has_a('francisca', 'ammunition_type', 'no_ammunition')
weapons_net.add_has_a('francisca', 'is_throwable', 'throwable')
weapons_net.add_has_a('francisca', 'era_used', '500,800')
weapons_net.add_has_a('francisca', 'has_part', 'axe_head')
weapons_net.add_has_a('francisca', 'historical_origin', 'francs')
weapons_net.add_has_a('francisca', 'length_inches', '18,21')
weapons_net.add_has_a('francisca', 'primary_material', 'wood')
weapons_net.add_has_a('francisca', 'secondary_material', 'iron')

weapons_net.add_is_a('gun', 'ranged_weapon')
weapons_net.add_has_a('gun', 'effective_range', 'long')
weapons_net.add_has_a('gun', 'attack_rate', 'slow')

weapons_net.add_is_a('javelin', 'ranged_weapon')
weapons_net.add_has_a('javelin', 'effective_range', 'medium')
weapons_net.add_has_a('javelin', 'attack_rate', 'slow')
weapons_net.add_has_a('javelin', 'ammunition_type', 'no_ammunition')
weapons_net.add_has_a('javelin', 'is_throwable', 'throwable')
weapons_net.add_has_a('javelin', 'alternate_name', 'light_spear')
weapons_net.add_has_a('javelin', 'alternate_name', 'france')
weapons_net.add_has_a('javelin', 'attack_type', 'puncture')
weapons_net.add_has_a('javelin', 'has_part', 'shaft')
weapons_net.add_has_a('javelin', 'has_part', 'head')
weapons_net.add_has_a('javelin', 'historical_origin', 'vikings')
weapons_net.add_has_a('javelin', 'length_inches', '72,96')

# Siege Weapon Categories
weapons_net.add_is_a('battering_ram', 'siege_weapon')
weapons_net.add_has_a('battering_ram', 'effective_against', 'fortifications')
weapons_net.add_has_a('battering_ram', 'effective_range', 'close_quarters')
weapons_net.add_has_a('battering_ram', 'attack_rate', 'medium')
weapons_net.add_has_a('battering_ram', 'movement_speed', 'medium')
weapons_net.add_has_a('battering_ram', 'ammunition_type', 'no_ammunition')

weapons_net.add_is_a('biological_weapon', 'siege_weapon')
weapons_net.add_has_a('biological_weapon', 'effective_against', 'personnel')
weapons_net.add_has_a('biological_weapon', 'effective_range', 'long')
weapons_net.add_has_a('biological_weapon', 'attack_rate', 'medium')
weapons_net.add_has_a('biological_weapon', 'is_throwable', 'throwable')

weapons_net.add_is_a('chemical_weapon', 'siege_weapon')
weapons_net.add_has_a('chemical_weapon', 'effective_against', 'personnel')
weapons_net.add_has_a('chemical_weapon', 'effective_range', 'long')
weapons_net.add_has_a('chemical_weapon', 'attack_rate', 'medium')
weapons_net.add_has_a('chemical_weapon', 'is_throwable', 'throwable')

weapons_net.add_is_a('psychological_weapon', 'siege_weapon')
weapons_net.add_has_a('psychological_weapon', 'effective_against', 'personnel')
weapons_net.add_has_a('psychological_weapon', 'effective_range', 'long')
weapons_net.add_has_a('psychological_weapon', 'attack_rate', 'medium')
weapons_net.add_has_a('psychological_weapon', 'movement_speed', 'medium')
weapons_net.add_has_a('psychological_weapon', 'ammunition_type', 'no_ammunition')

weapons_net.add_is_a('moveable_structure', 'siege_weapon')
weapons_net.add_has_a('moveable_structure', 'effective_against', 'fortifications')
weapons_net.add_has_a('moveable_structure', 'ammunition_type', 'no_ammunition')

### Level 3 - Types of Weapons
weapons_net.add_is_a('dagger', 'bladed_hand_weapon')
weapons_net.add_has_a('dagger', 'effective_range', 'close_quarters')
weapons_net.add_has_a('dagger', 'is_throwable', 'throwable')
weapons_net.add_has_a('dagger', 'attack_type', 'puncture')
weapons_net.add_has_a('dagger', 'has_edges', 'two_edges')
weapons_net.add_has_a('dagger', 'has_part', 'blade')
weapons_net.add_has_a('dagger', 'primary_material', 'steel')
weapons_net.add_has_a('dagger', 'used_by', 'commoners')
weapons_net.add_has_a('dagger', 'used_by', 'knights')

weapons_net.add_is_a('knife', 'bladed_hand_weapon')
weapons_net.add_has_a('knife', 'effective_range', 'close_quarters')
weapons_net.add_has_a('knife', 'is_throwable', 'throwable')
weapons_net.add_has_a('knife', 'has_edges', 'one_edge')
weapons_net.add_has_a('knife', 'has_part', 'blade')
weapons_net.add_has_a('knife', 'primary_material', 'steel')
weapons_net.add_has_a('knife', 'secondary_material', 'wood')

weapons_net.add_is_a('sword', 'bladed_hand_weapon')
weapons_net.add_has_a('sword', 'has_edges', 'two_edges')
weapons_net.add_has_a('sword', 'has_part', 'blade')
weapons_net.add_has_a('sword', 'has_part', 'hilt')
weapons_net.add_has_a('sword', 'has_part', 'pommel')
weapons_net.add_has_a('sword', 'primary_material', 'steel')

weapons_net.add_is_a('club', 'blunt_hand_weapon')
weapons_net.add_has_a('club', 'attack_type', 'bash')

weapons_net.add_is_a('mace', 'blunt_hand_weapon')
weapons_net.add_has_a('mace', 'cost', 'cheap')
weapons_net.add_has_a('mace', 'length_inches', '24,48')
weapons_net.add_has_a('mace', 'used_by', 'foot_soldiers')
weapons_net.add_has_a('mace', 'used_by', 'cavalry')

weapons_net.add_is_a('becs_de_corbin', 'polearm')
weapons_net.add_has_a('becs_de_corbin', 'era_used', '1300,1500')
weapons_net.add_has_a('becs_de_corbin', 'alternate_name', 'war_hammer')
weapons_net.add_has_a('becs_de_corbin', 'attack_type', 'puncture')

weapons_net.add_is_a('danish_axe', 'polearm')
weapons_net.add_has_a('danish_axe', 'is_throwable', 'throwable')
weapons_net.add_has_a('danish_axe', 'era_used', '700,1400')
weapons_net.add_has_a('danish_axe', 'alternate_name', 'broad_axe')
weapons_net.add_has_a('danish_axe', 'alternate_name', 'dane_axe')
weapons_net.add_has_a('danish_axe', 'attack_type', 'chop')
weapons_net.add_has_a('danish_axe', 'has_edges', 'one_edge')
weapons_net.add_has_a('danish_axe', 'has_part', 'shaft')
weapons_net.add_has_a('danish_axe', 'has_part', 'axe_head')
weapons_net.add_has_a('danish_axe', 'historical_origin', 'vikings')
weapons_net.add_has_a('danish_axe', 'length_inches', '48,72')

weapons_net.add_is_a('fauchard', 'polearm')
weapons_net.add_has_a('fauchard', 'length_inches', '72,84')

weapons_net.add_is_a('glaive', 'polearm')
weapons_net.add_has_a('glaive', 'era_used', '1000,1400')
weapons_net.add_has_a('glaive', 'attack_type', 'thrust')
weapons_net.add_has_a('glaive', 'has_part', 'blade')
weapons_net.add_has_a('glaive', 'has_part', 'shaft')
weapons_net.add_has_a('glaive', 'length_inches', '72,84')
weapons_net.add_has_a('glaive', 'used_by', 'foot_soldiers')
weapons_net.add_has_a('glaive', 'used_by', 'cavalry')

weapons_net.add_is_a('guisarme', 'polearm')
weapons_net.add_has_a('guisarme', 'era_used', '1000,1400')
weapons_net.add_has_a('guisarme', 'alternate_name', 'giserne')
weapons_net.add_has_a('guisarme', 'alternate_name', 'bisarme')
weapons_net.add_has_a('guisarme', 'attack_type', 'dismount')
weapons_net.add_has_a('guisarme', 'has_part', 'shaft')
weapons_net.add_has_a('guisarme', 'has_part', 'hook')
weapons_net.add_has_a('guisarme', 'used_by', 'foot_soldiers')

weapons_net.add_is_a('halberd', 'polearm')
weapons_net.add_has_a('halberd', 'era_used', '1300,1500')
weapons_net.add_has_a('halberd', 'alternate_name', 'swiss_voulge')
weapons_net.add_has_a('halberd', 'alternate_name', 'hellembart')
weapons_net.add_has_a('halberd', 'attack_type', 'chop')
weapons_net.add_has_a('halberd', 'attack_type', 'thrust')
weapons_net.add_has_a('halberd', 'has_edges', 'one_edge')
weapons_net.add_has_a('halberd', 'has_part', 'shaft')
weapons_net.add_has_a('halberd', 'has_part', 'axe_head')
weapons_net.add_has_a('halberd', 'has_part', 'spike')
weapons_net.add_has_a('halberd', 'historical_origin', 'swiss')

weapons_net.add_is_a('lance', 'polearm')
weapons_net.add_has_a('lance', 'effective_against', 'cavalry')
weapons_net.add_has_a('lance', 'alternate_name', 'cavalry_spear')
weapons_net.add_has_a('lance', 'attack_type', 'puncture')
weapons_net.add_has_a('lance', 'has_part', 'shaft')
weapons_net.add_has_a('lance', 'has_part', 'tip')
weapons_net.add_has_a('lance', 'has_part', 'vamplate')
weapons_net.add_has_a('lance', 'historical_origin', 'romans')
weapons_net.add_has_a('lance', 'length_inches', '118,157')
weapons_net.add_has_a('lance', 'used_by', 'knights')
weapons_net.add_has_a('lance', 'used_by', 'foot_soldiers')

weapons_net.add_is_a('maul', 'polearm')
weapons_net.add_has_a('maul', 'era_used', '1300,1500')
weapons_net.add_has_a('maul', 'attack_type', 'bash')
weapons_net.add_has_a('maul', 'has_part', 'shaft')
weapons_net.add_has_a('maul', 'has_part', 'hammer_head')
weapons_net.add_has_a('maul', 'used_by', 'archers')

weapons_net.add_is_a('modern_flanged_mace', 'polearm')

weapons_net.add_is_a('pike', 'polearm')
weapons_net.add_has_a('pike', 'effective_against', 'cavalry')
weapons_net.add_has_a('pike', 'effective_range', 'medium')
weapons_net.add_has_a('pike', 'era_used', '500,1700')
weapons_net.add_has_a('pike', 'alternate_name', 'geldon')
weapons_net.add_has_a('pike', 'attack_type', 'thrust')
weapons_net.add_has_a('pike', 'length_inches', '120,240')
weapons_net.add_has_a('pike', 'primary_material', 'wood')
weapons_net.add_has_a('pike', 'secondary_material', 'iron')
weapons_net.add_has_a('pike', 'secondary_material', 'steel')

weapons_net.add_is_a('quarterstaff', 'polearm')
weapons_net.add_has_a('quarterstaff', 'era_used', '500,1800')
weapons_net.add_has_a('quarterstaff', 'alternate_name', 'staff')
weapons_net.add_has_a('quarterstaff', 'attack_type', 'bash')
weapons_net.add_has_a('quarterstaff', 'cost', 'cheap')
weapons_net.add_has_a('quarterstaff', 'has_edges', 'no_edges')
weapons_net.add_has_a('quarterstaff', 'primary_material', 'wood')

weapons_net.add_is_a('spear', 'polearm')
weapons_net.add_has_a('spear', 'is_throwable', 'throwable')
weapons_net.add_has_a('spear', 'era_used', '500,1400')
weapons_net.add_has_a('spear', 'attack_type', 'thrust')
weapons_net.add_has_a('spear', 'has_part', 'shaft')
weapons_net.add_has_a('spear', 'has_part', 'head')
weapons_net.add_has_a('spear', 'historical_origin', 'stone_age')
weapons_net.add_has_a('spear', 'length_inches', '72,96')
weapons_net.add_has_a('spear', 'primary_material', 'wood')
weapons_net.add_has_a('spear', 'secondary_material', 'obsidian')
weapons_net.add_has_a('spear', 'secondary_material', 'iron')
weapons_net.add_has_a('spear', 'secondary_material', 'bronze')


###Level 4 - Leaves
weapons_net.add_is_a('flail', 'blunt_hand_weapon')

weapons_net.add_is_a('horsemans_pick', 'blunt_hand_weapon')

# Maces
weapons_net.add_is_a('flanged_mace', 'mace')
weapons_net.add_has_a('flanged_mace', 'era_used', '900,1300')
weapons_net.add_has_a('flanged_mace', 'alternate_name', 'bardoukion')
weapons_net.add_has_a('flanged_mace', 'attack_type', 'puncture')
weapons_net.add_has_a('flanged_mace', 'historical_origin', 'byzantine_empire')

weapons_net.add_is_a('holy_water_sprinkler', 'mace')
weapons_net.add_has_a('holy_water_sprinkler', 'alternate_name', 'goupillon')
weapons_net.add_has_a('holy_water_sprinkler', 'used_by', 'foot_soldiers')

weapons_net.add_is_a('plançon a picot', 'mace')
weapons_net.add_has_a('plançon a picot', 'era_used', '1300,1400')
weapons_net.add_has_a('plançon a picot', 'attack_type', 'puncture')
weapons_net.add_has_a('plançon a picot', 'historical_origin', 'milan')
weapons_net.add_has_a('plançon a picot', 'used_by', 'cavalieri')

# Clubs
weapons_net.add_is_a('morning_star', 'club')
weapons_net.add_has_a('morning_star', 'era_used', '1300,1400')
weapons_net.add_has_a('morning_star', 'used_by', 'foot_soldiers')
weapons_net.add_has_a('morning_star', 'used_by', 'peasant_militia')

# Danish Axes
weapons_net.add_is_a('pollaxe', 'danish_axe')
weapons_net.add_has_a('pollaxe', 'era_used', '1300,1500')

weapons_net.add_is_a('bardiche', 'pollaxe')

weapons_net.add_is_a('sparth', 'danish_axe')
weapons_net.add_has_a('sparth', 'era_used', '1200,1400')
weapons_net.add_has_a('sparth', 'alternate_name', 'pale_axe')
weapons_net.add_has_a('sparth', 'alternate_name', 'sparr')
weapons_net.add_has_a('sparth', 'has_part', 'broad_blade')
weapons_net.add_has_a('sparth', 'historical_origin', 'scotland')
weapons_net.add_has_a('sparth', 'historical_origin', 'ireland')

# Spears
weapons_net.add_is_a('winged_spear', 'spear')
weapons_net.add_has_a('winged_spear', 'is_throwable', 'throwable')
weapons_net.add_has_a('winged_spear', 'alternate_name', 'barred_spear')
weapons_net.add_has_a('winged_spear', 'has_part', 'shaft')
weapons_net.add_has_a('winged_spear', 'has_part', 'head')
weapons_net.add_has_a('winged_spear', 'has_part', 'wings')
weapons_net.add_has_a('winged_spear', 'historical_origin', 'romans')
weapons_net.add_has_a('winged_spear', 'historical_origin', 'francs')
weapons_net.add_has_a('winged_spear', 'historical_origin', 'vikings')

weapons_net.add_is_a('bohemian_ear_spoon', 'winged_spear')

weapons_net.add_is_a('spetum', 'winged_spear')

weapons_net.add_is_a('ranseur', 'corsque')

weapons_net.add_is_a('corsque', 'winged_spear')
weapons_net.add_has_a('corsque', 'alternate_name', 'chauve_souris')
weapons_net.add_has_a('corsque', 'attack_type', 'thrust')
weapons_net.add_has_a('corsque', 'has_part', 'three_bladed_head')

weapons_net.add_is_a('partisan', 'winged_spear')

weapons_net.add_is_a('angon', 'spear')
weapons_net.add_has_a('angon', 'effective_range', 'medium')
weapons_net.add_has_a('angon', 'is_throwable', 'throwable')
weapons_net.add_has_a('angon', 'attack_type', 'thrown')
weapons_net.add_has_a('angon', 'historical_origin', 'anglo_saxons')

# Bows
weapons_net.add_is_a('cross_bow', 'bow')
weapons_net.add_has_a('cross_bow', 'effective_range', 'medium')
weapons_net.add_has_a('cross_bow', 'ammunition_type', 'bolt')
weapons_net.add_has_a('cross_bow', 'era_used', '500,1500')
weapons_net.add_has_a('cross_bow', 'alternate_name', 'frankish_bow')
weapons_net.add_has_a('cross_bow', 'attack_type', 'puncture')

weapons_net.add_is_a('arbalest', 'cross_bow')
weapons_net.add_has_a('arbalest', 'era_used', '1300,1500')

weapons_net.add_is_a('long_bow', 'bow')
weapons_net.add_has_a('long_bow', 'effective_range', 'long')
weapons_net.add_has_a('long_bow', 'ammunition_type', 'arrow')
weapons_net.add_has_a('long_bow', 'era_used', '1200,1500')
weapons_net.add_has_a('long_bow', 'alternate_name', 'self_bow')
weapons_net.add_has_a('long_bow', 'attack_type', 'puncture')
weapons_net.add_has_a('long_bow', 'length_inches', '72')
weapons_net.add_has_a('long_bow', 'primary_material', 'wood')
weapons_net.add_has_a('long_bow', 'primary_material', 'yew_wood')

# Guns
weapons_net.add_is_a('arquebus', 'gun')
weapons_net.add_has_a('arquebus', 'ammunition_type', 'shot')
weapons_net.add_has_a('arquebus', 'ammunition_type', 'bullets')
weapons_net.add_has_a('arquebus', 'era_used', '1400,1600')
weapons_net.add_has_a('arquebus', 'alternate_name', 'hook_tube')
weapons_net.add_has_a('arquebus', 'attack_type', 'impact')
weapons_net.add_has_a('arquebus', 'has_part', 'barrel')
weapons_net.add_has_a('arquebus', 'has_part', 'stock')

weapons_net.add_is_a('musket', 'arquebus')

weapons_net.add_is_a('cavalier', 'arquebus')

weapons_net.add_is_a('hand_cannon', 'gun')
weapons_net.add_has_a('hand_cannon', 'effective_range', 'medium')
weapons_net.add_has_a('hand_cannon', 'ammunition_type', 'stone')
weapons_net.add_has_a('hand_cannon', 'ammunition_type', 'iron_ball')
weapons_net.add_has_a('hand_cannon', 'ammunition_type', 'arrow')
weapons_net.add_has_a('hand_cannon', 'era_used', '1100,1600')
weapons_net.add_has_a('hand_cannon', 'alternate_name', 'gonne')
weapons_net.add_has_a('hand_cannon', 'attack_type', 'impact')
weapons_net.add_has_a('hand_cannon', 'has_part', 'barrel')
weapons_net.add_has_a('hand_cannon', 'has_part', 'stock')
weapons_net.add_has_a('hand_cannon', 'historical_origin', 'china')
weapons_net.add_has_a('hand_cannon', 'primary_material', 'metal')
weapons_net.add_has_a('hand_cannon', 'secondary_material', 'wood')
weapons_net.add_has_a('hand_cannon', 'secondary_material', 'bamboo')

# Artillery
weapons_net.add_is_a('cannon', 'artillery')
weapons_net.add_has_a('cannon', 'ammunition_type', 'steel_ball')
weapons_net.add_has_a('cannon', 'attack_type', 'impact')

weapons_net.add_is_a('stone_thrower', 'artillery')
weapons_net.add_has_a('stone_thrower', 'ammunition_type', 'stone')
weapons_net.add_has_a('stone_thrower', 'alternate_name', 'pierrier')
weapons_net.add_has_a('stone_thrower', 'attack_type', 'impact')
weapons_net.add_has_a('stone_thrower', 'primary_material', 'wood')

# Swords
weapons_net.add_is_a('arming_sword', 'sword')
weapons_net.add_has_a('arming_sword', 'era_used', '1000,1350')
weapons_net.add_has_a('arming_sword', 'has_edges', 'two_edges')
weapons_net.add_has_a('arming_sword', 'length_inches', '30,32')
weapons_net.add_has_a('arming_sword', 'used_by', 'knight')

weapons_net.add_is_a('broad_sword', 'sword')
weapons_net.add_has_a('broad_sword', 'era_used', '1600,1800')
weapons_net.add_has_a('broad_sword', 'attack_type', 'thrust')
weapons_net.add_has_a('broad_sword', 'has_edges', 'two_edges')
weapons_net.add_has_a('broad_sword', 'has_part', 'basket_hilt')
weapons_net.add_has_a('broad_sword', 'length_inches', '30,32')

weapons_net.add_is_a('falchion', 'sword')
weapons_net.add_has_a('falchion', 'era_used', '1000,1500')
weapons_net.add_has_a('falchion', 'has_edges', 'one_edge')
weapons_net.add_has_a('falchion', 'historical_origin', 'europe')
weapons_net.add_has_a('falchion', 'length_inches', '30,32')
weapons_net.add_has_a('falchion', 'used_by', 'mounted_knight')

weapons_net.add_is_a('long_sword', 'sword')
weapons_net.add_has_a('long_sword', 'era_used', '1250,1550')
weapons_net.add_has_a('long_sword', 'alternate_name', 'bastard_sword')
weapons_net.add_has_a('long_sword', 'alternate_name', 'great_sword')
weapons_net.add_has_a('long_sword', 'alternate_name', 'hand_and_a_half_sword')
weapons_net.add_has_a('long_sword', 'attack_type', 'thrust')
weapons_net.add_has_a('long_sword', 'has_edges', 'two_edges')
weapons_net.add_has_a('long_sword', 'has_part', 'fullers_aka_blood_grooves')
weapons_net.add_has_a('long_sword', 'historical_origin', 'europe')
weapons_net.add_has_a('long_sword', 'length_inches', '40,48')
weapons_net.add_has_a('long_sword', 'used_by', 'knight')
weapons_net.add_has_a('long_sword', 'weight_kg', '1.2,2.4')

# Falchions
weapons_net.add_is_a('cleaver_falchion', 'falchion')
weapons_net.add_is_a('cusped_falchion', 'falchion')


# Daggers
weapons_net.add_is_a('anelace', 'dagger')
weapons_net.add_has_a('anelace', 'length_inches', '20,30')

weapons_net.add_is_a('poingnard', 'dagger')
weapons_net.add_has_a('poingnard', 'era_used', '1100,1800')
weapons_net.add_has_a('poingnard', 'historical_origin', 'france')

weapons_net.add_is_a('rondel', 'dagger')
weapons_net.add_has_a('rondel', 'length_inches', '20')

weapons_net.add_is_a('stiletto', 'dagger')
weapons_net.add_has_a('stiletto', 'era_used', '1100,1800')
weapons_net.add_has_a('stiletto', 'alternate_name', 'misericorde')
weapons_net.add_has_a('stiletto', 'attack_type', 'puncture')
weapons_net.add_has_a('stiletto', 'has_edges', 'no_edges')
weapons_net.add_has_a('stiletto', 'historical_origin', 'england')
weapons_net.add_has_a('stiletto', 'historical_origin', 'germany')

# Cannons
weapons_net.add_is_a('bombard', 'cannon')
weapons_net.add_has_a('bombard', 'ammunition_type', 'stone_ball')
weapons_net.add_has_a('bombard', 'era_used', '1350,1500')

weapons_net.add_is_a('petard', 'cannon')
weapons_net.add_has_a('petard', 'effective_against', 'walls')
weapons_net.add_has_a('petard', 'effective_against', 'gates')
weapons_net.add_has_a('petard', 'effective_against', 'tunnels')
weapons_net.add_has_a('petard', 'ammunition_type', 'iron_shot')
weapons_net.add_has_a('petard', 'era_used', '1500,1600')
weapons_net.add_has_a('petard', 'people_reqd', 'one_person')
weapons_net.add_has_a('petard', 'alternate_name', 'a_bomb')
weapons_net.add_has_a('petard', 'attack_type', 'explosion')
weapons_net.add_has_a('petard', 'historical_origin', 'france')

# Stone Throwers
weapons_net.add_is_a('ballista', 'stone_thrower')
weapons_net.add_has_a('ballista', 'ammunition_type', 'iron_tipped_bolt')
weapons_net.add_has_a('ballista', 'era_used', '1200,1500')

weapons_net.add_is_a('counterweight_trebuchet', 'stone_thrower')
weapons_net.add_has_a('counterweight_trebuchet', 'attack_rate', 'fast')
weapons_net.add_has_a('counterweight_trebuchet', 'era_used', '1100,1400')
weapons_net.add_has_a('counterweight_trebuchet', 'historical_origin', 'middle_east')

weapons_net.add_is_a('mangonel', 'stone_thrower')
weapons_net.add_has_a('mangonel', 'ammunition_type', 'multiple_stones')
weapons_net.add_has_a('mangonel', 'ammunition_type', 'burning_pitch')
weapons_net.add_has_a('mangonel', 'ammunition_type', 'flammable_pots')
weapons_net.add_has_a('mangonel', 'era_used', '1200,1500')
weapons_net.add_has_a('mangonel', 'has_part', 'fixed_bowl')

weapons_net.add_is_a('onager', 'stone_thrower')
weapons_net.add_has_a('onager', 'effective_range', 'medium')
weapons_net.add_has_a('onager', 'ammunition_type', 'stones')
weapons_net.add_has_a('onager', 'ammunition_type', 'burning_pitch')
weapons_net.add_has_a('onager', 'has_part', 'sling_basket')
weapons_net.add_has_a('onager', 'historical_origin', 'romans')

weapons_net.add_is_a('springald', 'stone_thrower')
weapons_net.add_has_a('springald', 'effective_range', 'medium')
weapons_net.add_has_a('springald', 'ammunition_type', 'heavy_bolt')
weapons_net.add_has_a('springald', 'era_used', '1200,1500')

weapons_net.add_is_a('traction_trebuchet', 'stone_thrower')
weapons_net.add_has_a('traction_trebuchet', 'era_used', '400,1400')
weapons_net.add_has_a('traction_trebuchet', 'people_reqd', '15_45')
weapons_net.add_has_a('traction_trebuchet', 'historical_origin', 'china')

# Siege Weapons
weapons_net.add_is_a('cat_and_weasel', 'moveable_structure')
weapons_net.add_has_a('cat_and_weasel', 'effective_range', 'close_quarters')
weapons_net.add_has_a('cat_and_weasel', 'attack_rate', 'slow')
weapons_net.add_has_a('cat_and_weasel', 'movement_speed', 'medium')

weapons_net.add_is_a('siege_tower', 'moveable_structure')
weapons_net.add_has_a('siege_tower', 'effective_range', 'short')
weapons_net.add_has_a('siege_tower', 'attack_rate', 'slow')


### Level 5 - Specific Instances / Famous Weapons
weapons_net.add_is_a('medici_falchion', 'falchion')
weapons_net.add_has_a('medici_falchion', 'use_case', 'ceremonial')
weapons_net.add_has_a('medici_falchion', 'era_used', '1560')
weapons_net.add_has_a('medici_falchion', 'decoration', 'coat_of_arms_of_cosimo_de_medici')
weapons_net.add_has_a('medici_falchion', 'historical_origin', 'florence')
weapons_net.add_has_a('medici_falchion', 'secondary_material', 'gold_plated')

weapons_net.add_is_a('the_wallace_sword', 'long_sword')
weapons_net.add_has_a('the_wallace_sword', 'era_used', '1298')
weapons_net.add_has_a('the_wallace_sword', 'historical_origin', 'scotland')
weapons_net.add_has_a('the_wallace_sword', 'used_by', 'william_wallace')
weapons_net.add_has_a('the_wallace_sword', 'weight_kg', '2.7')

weapons_net.add_is_a('mace_of_bishop_odo', 'mace')
weapons_net.add_has_a('mace_of_bishop_odo', 'era_used', '1066')
weapons_net.add_has_a('mace_of_bishop_odo', 'used_by', 'bishop_odo_of_bayeux')

weapons_net.add_is_a('almace', 'arming_sword')
weapons_net.add_has_a('almace', 'era_used', '778')
weapons_net.add_has_a('almace', 'used_by', 'archbishop_turpin')

weapons_net.add_is_a('royal_armouries_holy_water_sprinkler', 'holy_water_sprinkler')
weapons_net.add_has_a('royal_armouries_holy_water_sprinkler', 'use_case', 'ceremonial')
weapons_net.add_has_a('royal_armouries_holy_water_sprinkler', 'length_inches', '74.5')

weapons_net.add_is_a('wallace_collection_morning_star', 'morning_star')
weapons_net.add_has_a('wallace_collection_morning_star', 'use_case', 'ceremonial')
weapons_net.add_has_a('wallace_collection_morning_star', 'era_used', '1500,1600')
weapons_net.add_has_a('wallace_collection_morning_star', 'primary_material', 'steel')
weapons_net.add_has_a('wallace_collection_morning_star', 'secondary_material', 'gold,silver')


The network can be exported to JSON and Prolog files.

The Prolog export includes whichever rules file is specified.

In [11]:
# Export to JSON
weapons_net.export_to_json(filename='2-Medieval_Weapons_Network.json')

# Export to Prolog
weapons_net.export_to_prolog(filename='3-Medieval_Weapons_Network.pl', rules_filename='1-Semantic_Network_Prolog_Rules.pl')


Successfully saved: 262 nodes to 2-Medieval_Weapons_Network.json
Wrote File: Prolog written to 3-Medieval_Weapons_Network.pl.
 - wrote 84 IS_A facts
 - wrote 341 HAS_A facts


Then the network data is visualised in a dynamic node based network and a hierarchical diagram.<br>
Two alternative palettes are defined, one for print and another for on screen viewing.

Two digrams optimised for on-screen viewing have been uploaded to:<br>
https://datascience.daemonchild.com/year3/dat6002/Medieval_Weapons_Node_Diagram.html<br>
and<br>
https://datascience.daemonchild.com/year3/dat6002/Medieval_Weapons_Hierarchy.html<br>

Usage: <br>
- Use the mouse wheel to zoom.
- Drag empty space on the diagram to move around.
- Drag nodes to move them.


NB: PyVis requires Internet access to load the JavaScript libraries.

In [12]:

on_screen_palette = {'bg':"#000000", 
                     'text': "#ffffff", 
                     'edge_label': "#F0D90A", 
                     'edge_struct': "#c4830b81", 
                     'edge_prop': "#7847ff81",
                     0:"#C3ADFF", 
                     1:"#9670ff", 
                     2:"#7847ff", 
                     3:"#5A1FFF", 
                     4:"#4100F5", 
                     5:"#3100B8", 
                     6:"#22007C"}

print_palette =     {'bg':"#ffffff", 
                     'text': "#000000", 
                     'edge_label': "#111111", 
                     'edge_struct': "#333333", 
                     'edge_prop': "#666666",
                     0:"#dddddd", 
                     1:"#bbbbbb", 
                     2:"#aaaaaa", 
                     3:"#999999", 
                     4:"#888888", 
                     5:"#777777", 
                     6:"#666666"}

# Draw and Export PyVis diagrams

weapons_net.draw_dynamic(filename='4-Medieval_Weapons_Node_Diagram.html',show_levels=True, palette=on_screen_palette)
weapons_net.draw_hierarchical(filename='4-Medieval_Weapons_Hierarchy.html',show_levels=True, palette=on_screen_palette)
weapons_net.draw_dynamic(filename='4-Medieval_Weapons_Network_Node_Diagram_Printable.html',show_levels=True, palette=print_palette)
weapons_net.draw_hierarchical(filename='4-Medieval_Weapons_Hierarchy_Printable.html',show_levels=True, palette=print_palette)

# Draw focused diagrams (for the assignment report)
weapons_net.draw_hierarchical(filename='4-Medieval_Weapons_Hierarchy_focused_1.html',show_levels=True, palette=print_palette, filter_list=['sword'])
weapons_net.draw_hierarchical(filename='4-Medieval_Weapons_Hierarchy_focused_2.html',show_levels=True, palette=print_palette, filter_list=['stiletto','the_wallace_sword'])
weapons_net.draw_hierarchical(filename='4-Medieval_Weapons_Hierarchy_focused_3.html',show_levels=True, palette=print_palette, filter_list=['counterweight_trebuchet','siege_weapon'])

Wrote File: Network diagram written to 4-Medieval_Weapons_Node_Diagram.html.
Wrote File: Network diagram written to 4-Medieval_Weapons_Hierarchy.html.
Wrote File: Network diagram written to 4-Medieval_Weapons_Network_Node_Diagram_Printable.html.
Wrote File: Network diagram written to 4-Medieval_Weapons_Hierarchy_Printable.html.
Wrote File: Network diagram written to 4-Medieval_Weapons_Hierarchy_focused_1.html.
Wrote File: Network diagram written to 4-Medieval_Weapons_Hierarchy_focused_2.html.
Wrote File: Network diagram written to 4-Medieval_Weapons_Hierarchy_focused_3.html.


Showing that a prolog file can be imported into a new network:

In [13]:
# Create network
new_weapons_net = SemanticNetwork (root_node='weapon', title="Medieval Weapons (New)")

# Import Prolog File
new_weapons_net.import_from_prolog(filename='3-Medieval_Weapons_Network.pl')

Imported network from prolog file: 3-Medieval_Weapons_Network.pl
 - Added 84 IS_A facts
 - Added 341 HAS_A facts



# Prolog Rules for Semantic Networks

This sections explains how to use the provided rules to query a semantic network. 

## Facts
The rules are based on a network that is defined using the following fact structure:

**IS-A Facts**
`is_a ( child, parent )`
The relationships between items in the network are defined in a parent-child hierarchy using is_a.<br>
This builds a tree of relationships. 

Examples:

    is_a( melee_weapon, weapon ).
    is_a( bladed_hand_weapon, melee_weapon ).
    is_a( sword, bladed_hand_weapon ).

**HAS-A Facts**
The properties of an object are defined as has_a facts.
 
An *attribute* is a possible general characteristic known to the network, while a *property* is an instance of an attribute that is attached to a distinct item.

Items that are closer to the root of the relationship tree have fewer defined properties, and any such properties are more general. Conversely, objects that are leaves are likely to have more properties. Items inherit properties from objects that are closer to the root; these can be overridden (hidden) by more specific properties.

Examples:

    has_a( melee_weapon, effective_against, personnel ).
    has_a( bladed_hand_weapon, effective_range,close_quarters ).
    has_a( sword, has_part, blade ).
    has_a( sword, has_part, hilt ).
    has_a( sword, has_part, pommel ).

Querying the network for the unique attributes defined for a generic sword would therefore return the following, with the attributes being inherited from further up the tree:

 - has_part
 - effective_range
 - effective_against

## Rules

It is always possible to as Prolog to answer questions about the facts directly, for example `is_a (sword, ranged_weapon).` <br>
This would be False.

In additional, the rules provided for querying a network structured as defined above are as follows. These functions are generic, and could query any suitably defined network, whatever the subject matter.

 - is_a_member		- check class/parent membership 
 - has_a_property  - does the item have a property
 - list_all_items - list all the items in the network (pretty output)
 - get_all_items - get all the items in the network into sorted list
 - list_all_attributes - list all the attributes in the network (pretty output)
 - get_all_attributes - get all the attributes in the network into sorted list
 - list_properties    - list the defined properties for an item (including inherited)
 - list_items_with_property - list all items that have a specific property-value pair (pretty output)
 - show_tree - list items in an attractive tree

The following functions are specific to the Medieval Weapons Semantic Network:
 - longer_range - compares two weapons for range
 - list_weapons_by_year - find all weapons in use in a specific year



### is_a_member/2
Check class/parent membership. 

    % Base case
    is_a_member(Item, Class) :- is_a(Item, Class).
    
    % Recursive case
    is_a_member(Item, Class) :-
        is_a(Item, Parent),
        is_a_member(Parent, Class).

### has_a_property/3
Does the item have a property and/or value?

    % Base Case
    has_a_property(Item, Property, Value) :- has_a(Item, Property, Value).
    
    % Recursive Case
    has_a_property(Item, Property, Value) :-
		    is_a(Item, Class), 						% Check Class membership
		    has_a_property(Class, Property, Value), % Check parent class for properties
        \+ has_a(Item, Property, _). 				% Allow overrides


### list_all_items/0

List all possible items in the network in a pretty way. This should be functionally the same as the following command run against the Prolog source file:<br>
`cat prolog_file.pl | grep -e '^is_a(' | cut -f 2 -d "," | tr -d " " | sort -u`<br>
This rule also counts the total (equivalent to using `wc`).

    list_all_items  :-
		    get_all_items(List),
		    length(List, Count),
		    format('All items defined in the network:~n'),
   		    % Print the output header
   		    write('----------------------------------------'), nl,
   		    % Print each line of output
   		    forall(member(Item, List), format('- ~w~n', [Item])), nl,
   		    % Print count
   		    format('Total: ~w~n', [Count]).

References:

- https://www.swi-prolog.org/pldoc/man?predicate=format/2
- https://www.swi-prolog.org/pldoc/man?predicate=forall/2
- https://www.swi-prolog.org/pldoc/man?predicate=length/2


### get_all_items/1
Get all possible items in the network, returned into a sorted de-duplicated list. This rule is used by list_all_items.

    get_all_items(SortedList) :-
		    findall(Item, is_a(Item, _), List),
		    sort(List, SortedList).

References:

- https://www.swi-prolog.org/pldoc/man?predicate=findall/3
- https://www.swi-prolog.org/pldoc/man?predicate=sort/2


### list_all_attributes/0

List all known attributes in the network in a pretty manner.<br>
This should be functionally the same as the following command run against the Prolog source file:<br>
`cat prolog_file.pl | grep -e '^has_a(' | cut -f 2 -d "," | tr -d " " | sort -u`<br>
This rule also counts the total (equivalent to using `wc`).<br>

    list_all_attributes  :- 
		    get_all_possible_properties(List),
		    length(List, Count),
		    format('All attributes defined in the network:~n'),
		    write('----------------------------------------'), nl,
		    forall(member(Property, List), format('- ~w~n', [Property])), nl,
		    format('Total: ~w~n', [Count]).


### get_all_attributes/1

Get all possible property types in the network, returned in a list. Used by list_all_attributes.

    get_all_attributes(SortedList) :-
		    findall(Attribute, has_a(_, Attribute, _), List),
		    sort(List, SortedList).


### list_properties/1
List the defined properties for an item (including inherited) in a pretty way. 

    list_properties(Item) :-
		    % Collect all A-B pairs into a list ()
		    findall(A-B, has_a_property(Item, A, B), List),
	        % Sort this list
		    sort(List, SortedList),
		    % Print the output header
	        format('Properties for ~w:~n', [Item]),
	        write('----------------------------------------'), nl,
		    forall(member(A-B, SortedList), format('- ~w = ~w~n', [A, B])).


 
### list_items_with_property/2
Finds all items that have a specific property-value pair (including inherited), and prints out in a pretty manner.

    list_items_with_property(Prop, Value) :-
		    findall(Item, has_a_property(Item, Prop, Value), List),
		    sort(List, Sorted),
		    format('Items with Property ~w = ~w:~n', [Prop, Value]),
		    write('----------------------------------------'), nl,
		    forall(member(I, Sorted), format('- ~w~n', [I])).

### show_tree/1
Draws a nice tree from a given root position.

Example:

    show_tree(weapon).
    -- weapon
	    -- melee_weapon
		    -- bladed_hand_weapon
			    -- dagger
				    -- anelace
				    ..etc..

It is only intended that the first function is run directly:

    show_tree(Root) :- show_tree(Root, 0).

    show_tree(Item, Depth) :-
		    Indent is Depth  *  3,
		    tab(Indent),
		    format('-- ~w ~n', [Item]),
		    NewDepth is Depth  +  1,
		    % Find all children and recursively print them
		    forall(is_a(Child, Item), show_tree(Child, NewDepth)).

### longer_range/2
 
This functions compares two weapons based on range. As the effective_range values are text based ('long' range), weights are used to convert these values to integers which can be directly compared.

    longer_range(Weapon1, Weapon2) :-
		    has_a_property(Weapon1, effective_range, Value1),
		    has_a_property(Weapon2, effective_range, Value2),
		    % Get weights for each
		    range_to_weight(Value1, Weight1),
		    range_to_weight(Value2, Weight2),
		    % Compare
		    Weight1  >  Weight2.

The above function uses the following additional facts which define weights for the text values used.<br>
The higher the number, the longer the range.

    range_to_weight(long, 3).
    range_to_weight(medium, 2).
    range_to_weight(short, 1).
    range_to_weight(close_quarters, 0).


###  list_weapons_by_year/1

Find all weapons in use in a specific year. This function uses three helper functions to deal with the various cases present in the network.

    list_weapons_in_use_year(Year) :-
		    % Find all pairs of Item-Era where the Item was active in the given Year
		    findall(Weapon-Era, (is_in_era(Weapon, Year), has_a_property(Weapon, era_used, Era)), List),
		    sort(List, Sorted),
		    format('Weapons active in the year ~w:~n', [Year]),
		    write('----------------------------------------'), nl,
		    % Split the Item and Era for formatting
		    forall(member(I-E, Sorted), format('- ~w (~w)~n', [I, E])).


Returns True if Year falls within the Era defined for the weapon:

    is_in_era(Weapon, Year) :-
    		has_a_property(Weapon, era_used, Era),
    		check_year(Year, Era).

 
Exact match case, for example `has_a(almace, era_used, 778)`

    check_year(Year, Year) :- number(Year).
  
Range match case, for example `has_a(arbalest, era_used, (1300, 1500))`

    check_year(Year, (Start, End)) :-
		    Year  >=  Start,
		    Year  =<  End.

  



**Notebook Ends Here**